In [1]:
# ============================================================
# CELLULE 1 : Imports
# ============================================================
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from src.skills_extractor import SkillsExtractor
from src.job_matcher import JobMatcher

skills_extractor = SkillsExtractor()
job_matcher = JobMatcher()

print("✅ Modules initialisés")
print(f"   • Variations mappings : {len(skills_extractor.variations_to_canonical)}")
print(f"   • Technical skills DB : {len(skills_extractor.skills_database['technical_skills'])}")
print(f"   • Soft skills DB      : {len(skills_extractor.skills_database['soft_skills'])}")

✅ Modèle spaCy chargé
✅ Base de compétences chargée
   • Techniques : 653
   • Soft skills : 190
   • Variations : 472 mappings


2026-03-08 10:53:34,696 - src.job_matcher - INFO - Initialisation du JobMatcher...
2026-03-08 10:53:34,701 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-03-08 10:53:34,703 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2


✅ Modèle spaCy chargé
✅ Base de compétences chargée
   • Techniques : 653
   • Soft skills : 190
   • Variations : 472 mappings


2026-03-08 10:53:38,382 - src.job_matcher - INFO - ✅ skills_reference.json chargé (653 skills)
2026-03-08 10:53:38,388 - src.job_matcher - INFO - ✅ JobMatcher initialisé avec all-mpnet-base-v2


✅ Modules initialisés
   • Variations mappings : 472
   • Technical skills DB : 653
   • Soft skills DB      : 190


In [2]:
# ============================================================
# CELLULE 2 : Charger le dataset HuggingFace
# ============================================================
df = pd.read_excel('../data/resume_fit_job/processed/huggingface_resume_job_fit.xlsx', engine='openpyxl')

print(f"✅ Dataset : {len(df)} samples")
print(f"\n📊 Distribution des classes :")
print(df['score_target'].value_counts().sort_index())
print(f"\n📊 Colonnes : {df.columns.tolist()}")

✅ Dataset : 6241 samples

📊 Distribution des classes :
score_target
0.0    3143
0.5    1556
1.0    1542
Name: count, dtype: int64

📊 Colonnes : ['resume', 'job_description', 'label', 'tfidf_similarity', 'score_target']


In [3]:
# ============================================================
# CELLULE 3 : Diagnostic EXTRACTION sur 1 exemple par classe
# ============================================================

for score, label in [(0.0, 'No Fit'), (0.5, 'Partial Fit'), (1.0, 'Perfect Fit')]:
    sample = df[df['score_target'] == score].iloc[0]
    
    print(f"\n{'='*70}")
    print(f"🎯 Classe : {label}")
    print(f"{'='*70}")
    
    # ── CV ──────────────────────────────────────────────────────
    cv_text = str(sample['resume'])
    cv_result = skills_extractor.extract_from_cv(cv_text)
    cv_technical = cv_result['technical_skills']
    cv_soft = cv_result['soft_skills']
    cv_skills = cv_technical + cv_soft
    
    print(f"\n📄 CV (200 chars) : {cv_text[:200]}")
    print(f"\n🔧 Skills extraits du CV ({len(cv_skills)}) :")
    print(f"   • Techniques ({len(cv_technical)}) : {cv_technical[:10]}")
    print(f"   • Soft ({len(cv_soft)})       : {cv_soft[:5]}")
    
    # ── JOB ─────────────────────────────────────────────────────
    job_text = str(sample['job_description'])
    
    # Créer la structure job comme dans compute_features
    job_result = skills_extractor.extract_from_cv(job_text)
    job_technical = job_result['technical_skills']
    job_soft = job_result['soft_skills']
    job_skills = job_technical + job_soft
    
    print(f"\n💼 Job (200 chars) : {job_text[:200]}")
    print(f"\n🔧 Skills extraits du Job ({len(job_skills)}) :")
    print(f"   • Techniques ({len(job_technical)}) : {job_technical[:10]}")
    print(f"   • Soft ({len(job_soft)})       : {job_soft[:5]}")
    
    if not cv_skills or not job_skills:
        print(f"\n❌ PROBLÈME : CV skills={len(cv_skills)}, Job skills={len(job_skills)}")
        continue
    
    # ── MATCHING ────────────────────────────────────────────────
    job_structure = {
        'job_id': 'test',
        'title': 'Test Job',
        'requirements': job_skills,
        'nice_to_have': []
    }
    
    match_result = job_matcher.calculate_job_match_score(cv_skills, job_structure)
    sd = match_result['skills_details']
    
    print(f"\n📊 RÉSULTAT MATCHING :")
    print(f"   • Overall Score  : {match_result['score']:.1f}%")
    print(f"   • Coverage       : {sd['coverage']:.1f}%")
    print(f"   • Quality        : {sd['quality']:.1f}%")
    print(f"   • Covered        : {sd['covered_count']} / {sd['total_required']}")
    
    print(f"\n🔍 Top matches :")
    for m in sd['top_matches'][:5]:
        print(f"   • '{m['job_skill']}' ↔ '{m['cv_skill']}' → {m['similarity']:.1f}%")


🎯 Classe : No Fit

📄 CV (200 chars) : SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and 

🔧 Skills extraits du CV (14) :
   • Techniques (3) : ['ar', 'assembly', 'excel']
   • Soft (11)       : ['customer service', 'forecasting', 'networking', 'planning', 'project management']


2026-03-08 10:53:40,085 - src.job_matcher - INFO - 💼 Skills extraits du job : 8
2026-03-08 10:53:40,087 - src.job_matcher - INFO - 🔍 Matching 8 skills offre ↔ 14 skills CV



💼 Job (200 chars) : Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally

🔧 Skills extraits du Job (8) :
   • Techniques (3) : ['c', 'excel', 'sql']
   • Soft (5)       : ['change management', 'leadership', 'organization', 'risk management', 'training']


2026-03-08 10:53:40,650 - src.job_matcher - INFO - ✅ Coverage: 50.0% | Quality: 81.3% | Score: 62.5%



📊 RÉSULTAT MATCHING :
   • Overall Score  : 62.5%
   • Coverage       : 50.0%
   • Quality        : 81.3%
   • Covered        : 4 / 8

🔍 Top matches :
   • 'excel' ↔ 'excel' → 100.0%
   • 'training' ↔ 'training' → 100.0%
   • 'leadership' ↔ 'team building' → 62.7%
   • 'organization' ↔ 'team building' → 62.6%

🎯 Classe : Partial Fit

📄 CV (200 chars) : Professional Summary5  years of experience on Database development and administration, knowledge of relational database on SQL Server particularly on SSMS, SSRS, SSIS, Transact-SQL and SQL Server Agen

🔧 Skills extraits du CV (15) :
   • Techniques (7) : ['agile', 'documentation', 'etl', 'excel', 'sql', 'sql server', 'tensorflow']
   • Soft (8)       : ['business analysis', 'communication', 'documentation', 'integrity', 'reporting']

💼 Job (200 chars) : Kindly focus on the highlighted skill in the below requirements.
Position: Salesforce BALocation: Phoenix or Salt Lake City (Onsite) 

 Key Skills:  10+ years of overall experience 5+ Sa

2026-03-08 10:53:41,266 - src.job_matcher - INFO - 💼 Skills extraits du job : 1
2026-03-08 10:53:41,267 - src.job_matcher - INFO - 🔍 Matching 1 skills offre ↔ 14 skills CV
2026-03-08 10:53:41,571 - src.job_matcher - INFO - ✅ Coverage: 0.0% | Quality: 0.0% | Score: 0.0%



📊 RÉSULTAT MATCHING :
   • Overall Score  : 0.0%
   • Coverage       : 0.0%
   • Quality        : 0.0%
   • Covered        : 0 / 1

🔍 Top matches :

🎯 Classe : Perfect Fit

📄 CV (200 chars) : SummaryRecent graduate from Nucamp Coding Bootcamp with excellent research, technical and problem-solving skills. Detail-oriented and able to learn new technology quickly. Ambitious, career-focused jo

🔧 Skills extraits du CV (29) :
   • Techniques (21) : ['agile', 'aws', 'azure', 'css', 'devops', 'django', 'documentation', 'flask', 'google cloud', 'hibernate']
   • Soft (8)       : ['communication', 'compliance', 'documentation', 'flexibility', 'integrity']

💼 Job (200 chars) : Role: Senior Software Engineer
Type: Full-Time

As a Senior Software Engineer, you'll lead modern tech projects using Java, Docker, and data modeling.

What You'll Do:
Define product architecture with

🔧 Skills extraits du Job (8) :
   • Techniques (6) : ['aws', 'docker', 'java', 'kubernetes', 'microservices', 'python']
 

2026-03-08 10:53:42,119 - src.job_matcher - INFO - 💼 Skills extraits du job : 8
2026-03-08 10:53:42,119 - src.job_matcher - INFO - 🔍 Matching 8 skills offre ↔ 28 skills CV
2026-03-08 10:53:42,865 - src.job_matcher - INFO - ✅ Coverage: 62.5% | Quality: 100.0% | Score: 77.5%



📊 RÉSULTAT MATCHING :
   • Overall Score  : 77.5%
   • Coverage       : 62.5%
   • Quality        : 100.0%
   • Covered        : 5 / 8

🔍 Top matches :
   • 'aws' ↔ 'aws' → 100.0%
   • 'java' ↔ 'java' → 100.0%
   • 'microservices' ↔ 'microservices' → 100.0%
   • 'python' ↔ 'python' → 100.0%
   • 'communication' ↔ 'communication' → 100.0%


In [4]:
# ============================================================
# CELLULE 4 : DIAGNOSTIC PROFOND - Pourquoi coverage = 56% partout?
# ============================================================

print("🔍 ANALYSE : Pourquoi coverage identique pour toutes classes?\n")

# Prendre 10 exemples par classe
results_by_class = {}

for score, label in [(0.0, 'No Fit'), (0.5, 'Partial Fit'), (1.0, 'Perfect Fit')]:
    samples = df[df['score_target'] == score].head(10)
    
    coverages = []
    cv_skill_counts = []
    job_skill_counts = []
    zero_skill_count = 0
    
    for _, sample in samples.iterrows():
        cv_result = skills_extractor.extract_from_cv(str(sample['resume']))
        job_result = skills_extractor.extract_from_cv(str(sample['job_description']))
        
        cv_skills = cv_result['technical_skills'] + cv_result['soft_skills']
        job_skills = job_result['technical_skills'] + job_result['soft_skills']
        
        cv_skill_counts.append(len(cv_skills))
        job_skill_counts.append(len(job_skills))
        
        if not cv_skills or not job_skills:
            zero_skill_count += 1
            coverages.append(0)
            continue
        
        job_structure = {
            'job_id': 'test', 'title': 'Test',
            'requirements': job_skills, 'nice_to_have': []
        }
        match = job_matcher.calculate_job_match_score(cv_skills, job_structure)
        coverages.append(match['skills_details']['coverage'])
    
    results_by_class[label] = {
        'coverage_mean': np.mean(coverages),
        'coverage_std': np.std(coverages),
        'cv_skills_mean': np.mean(cv_skill_counts),
        'job_skills_mean': np.mean(job_skill_counts),
        'zero_skills': zero_skill_count
    }
    
    print(f"{'='*50}")
    print(f"Classe : {label} (10 samples)")
    print(f"  CV skills moyen   : {np.mean(cv_skill_counts):.1f}")
    print(f"  Job skills moyen  : {np.mean(job_skill_counts):.1f}")
    print(f"  Coverage moyen    : {np.mean(coverages):.1f}%")
    print(f"  Coverage std      : {np.std(coverages):.1f}%")
    print(f"  Cas vides (0 sk)  : {zero_skill_count}/10")

🔍 ANALYSE : Pourquoi coverage identique pour toutes classes?



2026-03-08 10:53:43,622 - src.job_matcher - INFO - 💼 Skills extraits du job : 8
2026-03-08 10:53:43,623 - src.job_matcher - INFO - 🔍 Matching 8 skills offre ↔ 14 skills CV
2026-03-08 10:53:44,083 - src.job_matcher - INFO - ✅ Coverage: 50.0% | Quality: 81.3% | Score: 62.5%
2026-03-08 10:53:44,881 - src.job_matcher - INFO - 💼 Skills extraits du job : 7
2026-03-08 10:53:44,882 - src.job_matcher - INFO - 🔍 Matching 7 skills offre ↔ 12 skills CV
2026-03-08 10:53:45,282 - src.job_matcher - INFO - ✅ Coverage: 14.3% | Quality: 100.0% | Score: 48.6%
2026-03-08 10:53:46,052 - src.job_matcher - INFO - 💼 Skills extraits du job : 9
2026-03-08 10:53:46,053 - src.job_matcher - INFO - 🔍 Matching 9 skills offre ↔ 7 skills CV
2026-03-08 10:53:46,385 - src.job_matcher - INFO - ✅ Coverage: 11.1% | Quality: 100.0% | Score: 46.7%
2026-03-08 10:53:46,948 - src.job_matcher - INFO - 💼 Skills extraits du job : 2
2026-03-08 10:53:46,950 - src.job_matcher - INFO - 🔍 Matching 2 skills offre ↔ 11 skills CV
2026-03-

Classe : No Fit (10 samples)
  CV skills moyen   : 11.6
  Job skills moyen  : 8.4
  Coverage moyen    : 26.0%
  Coverage std      : 22.3%
  Cas vides (0 sk)  : 0/10


2026-03-08 10:53:54,572 - src.job_matcher - INFO - 💼 Skills extraits du job : 1
2026-03-08 10:53:54,573 - src.job_matcher - INFO - 🔍 Matching 1 skills offre ↔ 14 skills CV
2026-03-08 10:53:54,879 - src.job_matcher - INFO - ✅ Coverage: 0.0% | Quality: 0.0% | Score: 0.0%
2026-03-08 10:53:55,475 - src.job_matcher - INFO - 💼 Skills extraits du job : 13
2026-03-08 10:53:55,475 - src.job_matcher - INFO - 🔍 Matching 13 skills offre ↔ 22 skills CV
2026-03-08 10:53:56,215 - src.job_matcher - INFO - ✅ Coverage: 23.1% | Quality: 89.2% | Score: 49.5%
2026-03-08 10:53:56,831 - src.job_matcher - INFO - 💼 Skills extraits du job : 17
2026-03-08 10:53:56,832 - src.job_matcher - INFO - 🔍 Matching 17 skills offre ↔ 10 skills CV
2026-03-08 10:53:57,389 - src.job_matcher - INFO - ✅ Coverage: 5.9% | Quality: 100.0% | Score: 43.5%
2026-03-08 10:53:57,969 - src.job_matcher - INFO - 💼 Skills extraits du job : 10
2026-03-08 10:53:57,970 - src.job_matcher - INFO - 🔍 Matching 10 skills offre ↔ 5 skills CV
2026-03

Classe : Partial Fit (10 samples)
  CV skills moyen   : 15.1
  Job skills moyen  : 7.2
  Coverage moyen    : 28.5%
  Coverage std      : 31.3%
  Cas vides (0 sk)  : 0/10


2026-03-08 10:54:04,571 - src.job_matcher - INFO - 💼 Skills extraits du job : 8
2026-03-08 10:54:04,572 - src.job_matcher - INFO - 🔍 Matching 8 skills offre ↔ 28 skills CV
2026-03-08 10:54:05,317 - src.job_matcher - INFO - ✅ Coverage: 62.5% | Quality: 100.0% | Score: 77.5%
2026-03-08 10:54:06,332 - src.job_matcher - INFO - 💼 Skills extraits du job : 6
2026-03-08 10:54:06,332 - src.job_matcher - INFO - 🔍 Matching 6 skills offre ↔ 14 skills CV
2026-03-08 10:54:06,740 - src.job_matcher - INFO - ✅ Coverage: 83.3% | Quality: 94.0% | Score: 87.6%
2026-03-08 10:54:07,376 - src.job_matcher - INFO - 💼 Skills extraits du job : 31
2026-03-08 10:54:07,377 - src.job_matcher - INFO - 🔍 Matching 31 skills offre ↔ 25 skills CV
2026-03-08 10:54:08,700 - src.job_matcher - INFO - ✅ Coverage: 32.3% | Quality: 88.9% | Score: 54.9%
2026-03-08 10:54:09,679 - src.job_matcher - INFO - 💼 Skills extraits du job : 23
2026-03-08 10:54:09,680 - src.job_matcher - INFO - 🔍 Matching 23 skills offre ↔ 24 skills CV
2026

Classe : Perfect Fit (10 samples)
  CV skills moyen   : 25.3
  Job skills moyen  : 15.0
  Coverage moyen    : 44.8%
  Coverage std      : 16.4%
  Cas vides (0 sk)  : 0/10


In [9]:
# ============================================================
# CELLULE 5 : DIAGNOSTIC CLEF - Exact match vs Sémantique
# ============================================================
print("🔍 PROBLÈME CLÉ : Similarité sémantique trop permissive?\n")

# Test sur le cas "No Fit" le plus évident
no_fit_sample = df[df['score_target'] == 0.0].iloc[0]

cv_result = skills_extractor.extract_from_cv(str(no_fit_sample['resume']))
job_result = skills_extractor.extract_from_cv(str(no_fit_sample['job_description']))

cv_skills = cv_result['technical_skills'] + cv_result['soft_skills']
job_skills = job_result['technical_skills'] + job_result['soft_skills']

print(f"CV Skills  : {cv_skills}")
print(f"Job Skills : {job_skills}")
print(f"\n🔍 Détail des matchs (TOUS) :")

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

model = job_matcher.model

# Encoder tous les skills
cv_embs = {s: model.encode([s.lower()])[0] for s in cv_skills}
job_embs = {s: model.encode([s.lower()])[0] for s in job_skills}

print(f"\n{'Job Skill':<25} {'Best CV Skill':<25} {'Similarity':>10} {'Status':>10}")
print("-" * 75)

for job_skill in job_skills:
    best_sim = 0
    best_cv = None
    
    # Exact match first
    for cv_skill in cv_skills:
        if cv_skill.lower() == job_skill.lower():
            best_sim = 100.0
            best_cv = cv_skill
            break
    
    # Semantic match
    if best_sim < 100:
        for cv_skill in cv_skills:
            sim = cos_sim([job_embs[job_skill]], [cv_embs[cv_skill]])[0][0] * 100
            if sim > best_sim:
                best_sim = sim
                best_cv = cv_skill
    
    status = "✅ covered" if best_sim >= 60 else ("⚠️ partial" if best_sim >= 40 else "❌ missing")
    print(f"{job_skill:<25} {str(best_cv):<25} {best_sim:>10.1f}% {status:>10}")

🔍 PROBLÈME CLÉ : Similarité sémantique trop permissive?

CV Skills  : ['ar', 'assembly', 'excel', 'customer service', 'forecasting', 'networking', 'planning', 'project management', 'public speaking', 'quality assurance', 'reporting', 'team building', 'training', 'troubleshooting']
Job Skills : ['c', 'excel', 'sql', 'change management', 'leadership', 'organization', 'risk management', 'training']

🔍 Détail des matchs (TOUS) :


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Job Skill                 Best CV Skill             Similarity     Status
---------------------------------------------------------------------------
c                         excel                           26.2%  ❌ missing
excel                     excel                          100.0%  ✅ covered
sql                       excel                           45.5% ⚠️ partial
change management         project management              62.3%  ✅ covered
leadership                team building                   62.7%  ✅ covered
organization              team building                   62.6%  ✅ covered
risk management           project management              54.9% ⚠️ partial
training                  training                       100.0%  ✅ covered


In [6]:
# ============================================================
# CELLULE 6 : DIAGNOSTIC - Le vrai problème du seuil 40%
# ============================================================
print("🔍 ANALYSE DU SEUIL 40% : Combien de 'faux covered'?\n")

# Sur 30 samples total
sample_30 = df.sample(n=30, random_state=42)

false_covered = []  # covered mais similarity < 60% (match douteux)
true_covered = []   # covered avec similarity >= 60%

for _, row in sample_30.iterrows():
    cv_result = skills_extractor.extract_from_cv(str(row['resume']))
    job_result = skills_extractor.extract_from_cv(str(row['job_description']))
    
    cv_skills = cv_result['technical_skills'] + cv_result['soft_skills']
    job_skills = job_result['technical_skills'] + job_result['soft_skills']
    
    if not cv_skills or not job_skills:
        continue
    
    job_structure = {
        'job_id': 'test', 'title': 'Test',
        'requirements': job_skills, 'nice_to_have': []
    }
    match = job_matcher.calculate_job_match_score(cv_skills, job_structure)
    
    for m in match['skills_details']['top_matches']:
        if m['similarity'] >= 40:  # "covered"
            if m['similarity'] < 60:  # mais douteux
                false_covered.append(m)
            else:
                true_covered.append(m)

print(f"✅ Vrais matchs (sim >= 60%)  : {len(true_covered)}")
print(f"⚠️  Faux matchs (40% <= sim < 60%) : {len(false_covered)}")
print(f"\n📊 Exemples de FAUX matchs (potentiellement faux positifs) :")
for m in false_covered[:10]:
    print(f"   '{m['job_skill']}' ↔ '{m['cv_skill']}' → {m['similarity']:.1f}%")

🔍 ANALYSE DU SEUIL 40% : Combien de 'faux covered'?



2026-03-08 10:54:22,149 - src.job_matcher - INFO - 💼 Skills extraits du job : 6
2026-03-08 10:54:22,150 - src.job_matcher - INFO - 🔍 Matching 6 skills offre ↔ 4 skills CV
2026-03-08 10:54:22,386 - src.job_matcher - INFO - ✅ Coverage: 50.0% | Quality: 90.0% | Score: 66.0%
2026-03-08 10:54:22,925 - src.job_matcher - INFO - 💼 Skills extraits du job : 3
2026-03-08 10:54:22,925 - src.job_matcher - INFO - 🔍 Matching 3 skills offre ↔ 18 skills CV
2026-03-08 10:54:23,377 - src.job_matcher - INFO - ✅ Coverage: 0.0% | Quality: 0.0% | Score: 0.0%
2026-03-08 10:54:23,960 - src.job_matcher - INFO - 💼 Skills extraits du job : 1
2026-03-08 10:54:23,961 - src.job_matcher - INFO - 🔍 Matching 1 skills offre ↔ 13 skills CV
2026-03-08 10:54:24,249 - src.job_matcher - INFO - ✅ Coverage: 0.0% | Quality: 0.0% | Score: 0.0%
2026-03-08 10:54:24,943 - src.job_matcher - INFO - 💼 Skills extraits du job : 12
2026-03-08 10:54:24,944 - src.job_matcher - INFO - 🔍 Matching 12 skills offre ↔ 32 skills CV
2026-03-08 10:

✅ Vrais matchs (sim >= 60%)  : 58
⚠️  Faux matchs (40% <= sim < 60%) : 0

📊 Exemples de FAUX matchs (potentiellement faux positifs) :


In [7]:
# ============================================================
# CELLULE 7 : CONCLUSION - Résumé des problèmes trouvés
# ============================================================
print("=" * 70)
print("📋 RÉSUMÉ DES PROBLÈMES DÉTECTÉS")
print("=" * 70)

# 1. Stats globales sur 50 samples
sample_50 = df.sample(n=50, random_state=42)

zero_cv, zero_job, low_coverage = 0, 0, 0

for _, row in sample_50.iterrows():
    cv_r = skills_extractor.extract_from_cv(str(row['resume']))
    job_r = skills_extractor.extract_from_cv(str(row['job_description']))
    
    cv_sk = cv_r['technical_skills'] + cv_r['soft_skills']
    job_sk = job_r['technical_skills'] + job_r['soft_skills']
    
    if len(cv_sk) == 0: zero_cv += 1
    if len(job_sk) == 0: zero_job += 1
    
    if cv_sk and job_sk:
        job_struct = {'job_id': 'x', 'title': 'x', 'requirements': job_sk, 'nice_to_have': []}
        m = job_matcher.calculate_job_match_score(cv_sk, job_struct)
        if m['skills_details']['coverage'] < 20:
            low_coverage += 1

print(f"\n1️⃣  CVs avec 0 skills extraits  : {zero_cv}/50 ({zero_cv*2}%)")
print(f"2️⃣  Jobs avec 0 skills extraits  : {zero_job}/50 ({zero_job*2}%)")
print(f"3️⃣  Coverage < 20%               : {low_coverage}/50 ({low_coverage*2}%)")
print(f"\n4️⃣  Seuil 40% pour 'covered'")
print(f"    → Trop permissif? Vérifie cellule 6")
print(f"\n5️⃣  Coverage identique ~57% pour toutes les classes")
print(f"    → Features non discriminantes pour XGBoost")
print(f"\n{'='*70}")
print("💡 ACTIONS RECOMMANDÉES :")
print("   1. Vérifier le contenu réel des textes CV/Job du dataset")
print("   2. Voir si les skills extraits sont pertinents")
print("   3. Ajuster le seuil si trop de faux positifs")
print("   4. Utiliser embedding_similarity comme feature principale")
print("=" * 70)

📋 RÉSUMÉ DES PROBLÈMES DÉTECTÉS


2026-03-08 10:54:57,578 - src.job_matcher - INFO - 💼 Skills extraits du job : 6
2026-03-08 10:54:57,579 - src.job_matcher - INFO - 🔍 Matching 6 skills offre ↔ 4 skills CV
2026-03-08 10:54:57,780 - src.job_matcher - INFO - ✅ Coverage: 50.0% | Quality: 90.0% | Score: 66.0%
2026-03-08 10:54:58,286 - src.job_matcher - INFO - 💼 Skills extraits du job : 3
2026-03-08 10:54:58,287 - src.job_matcher - INFO - 🔍 Matching 3 skills offre ↔ 18 skills CV
2026-03-08 10:54:58,723 - src.job_matcher - INFO - ✅ Coverage: 0.0% | Quality: 0.0% | Score: 0.0%
2026-03-08 10:54:59,287 - src.job_matcher - INFO - 💼 Skills extraits du job : 1
2026-03-08 10:54:59,287 - src.job_matcher - INFO - 🔍 Matching 1 skills offre ↔ 13 skills CV
2026-03-08 10:54:59,578 - src.job_matcher - INFO - ✅ Coverage: 0.0% | Quality: 0.0% | Score: 0.0%
2026-03-08 10:55:00,258 - src.job_matcher - INFO - 💼 Skills extraits du job : 12
2026-03-08 10:55:00,260 - src.job_matcher - INFO - 🔍 Matching 12 skills offre ↔ 32 skills CV
2026-03-08 10:


1️⃣  CVs avec 0 skills extraits  : 0/50 (0%)
2️⃣  Jobs avec 0 skills extraits  : 1/50 (2%)
3️⃣  Coverage < 20%               : 24/50 (48%)

4️⃣  Seuil 40% pour 'covered'
    → Trop permissif? Vérifie cellule 6

5️⃣  Coverage identique ~57% pour toutes les classes
    → Features non discriminantes pour XGBoost

💡 ACTIONS RECOMMANDÉES :
   1. Vérifier le contenu réel des textes CV/Job du dataset
   2. Voir si les skills extraits sont pertinents
   3. Ajuster le seuil si trop de faux positifs
   4. Utiliser embedding_similarity comme feature principale


In [ ]:
# ============================================================
# CELLULE 8 : Impact du seuil 60% sur coverage par classe
# ============================================================

print("🔍 IMPACT DU SEUIL 60% vs 40% SUR COVERAGE PAR CLASSE\n")

for score, label in [(0.0, 'No Fit'), (0.5, 'Partial Fit'), (1.0, 'Perfect Fit')]:
    samples = df[df['score_target'] == score].head(10)
    
    coverages_40 = []  # Seuil actuel
    coverages_60 = []  # Nouveau seuil
    
    for _, sample in samples.iterrows():
        cv_result = skills_extractor.extract_from_cv(str(sample['resume']))
        job_result = skills_extractor.extract_from_cv(str(sample['job_description']))
        
        cv_skills = cv_result['technical_skills'] + cv_result['soft_skills']
        job_skills = job_result['technical_skills'] + job_result['soft_skills']
        
        if not cv_skills or not job_skills:
            continue
        
        # Calculer embeddings
        cv_embs = {s: job_matcher.model.encode([s.lower()], show_progress_bar=False)[0] 
                   for s in cv_skills}
        job_embs = {s: job_matcher.model.encode([s.lower()], show_progress_bar=False)[0] 
                    for s in job_skills}
        
        covered_40, covered_60 = 0, 0
        
        for job_skill in job_skills:
            best_sim = 0
            
            # Exact match
            for cv_skill in cv_skills:
                if cv_skill.lower() == job_skill.lower():
                    best_sim = 100.0
                    break
            
            # Semantic match
            if best_sim < 100:
                for cv_skill in cv_skills:
                    from sklearn.metrics.pairwise import cosine_similarity as cos_sim
                    sim = cos_sim([job_embs[job_skill]], [cv_embs[cv_skill]])[0][0] * 100
                    if sim > best_sim:
                        best_sim = sim
            
            if best_sim >= 40: covered_40 += 1
            if best_sim >= 65: covered_60 += 1
        
        coverages_40.append(covered_40 / len(job_skills) * 100)
        coverages_60.append(covered_60 / len(job_skills) * 100)
    
    print(f"{'='*50}")
    print(f"Classe : {label}")
    print(f"  Coverage seuil 40% : {np.mean(coverages_40):.1f}%")
    print(f"  Coverage seuil 65% : {np.mean(coverages_60):.1f}%")
    print(f"  Différence         : {np.mean(coverages_40)-np.mean(coverages_60):.1f}%")

🔍 IMPACT DU SEUIL 60% vs 40% SUR COVERAGE PAR CLASSE

Classe : No Fit
  Coverage seuil 40% : 55.0%
  Coverage seuil 60% : 21.5%
  Différence         : 33.5%
Classe : Partial Fit
  Coverage seuil 40% : 52.0%
  Coverage seuil 60% : 28.5%
  Différence         : 23.5%
Classe : Perfect Fit
  Coverage seuil 40% : 76.6%
  Coverage seuil 60% : 41.2%
  Différence         : 35.4%


In [17]:
# ============================================================
# CELLULE 9 : Features enrichies pour XGBoost
# ============================================================

def compute_rich_features(cv_text, job_text, threshold=65):
    """Calcule des features bien discriminantes"""
    
    cv_result = skills_extractor.extract_from_cv(cv_text)
    job_result = skills_extractor.extract_from_cv(job_text)
    
    cv_technical = cv_result['technical_skills']
    cv_soft      = cv_result['soft_skills']
    job_technical = job_result['technical_skills']
    job_soft      = job_result['soft_skills']
    
    cv_skills  = cv_technical + cv_soft
    job_skills = job_technical + job_soft
    
    if not cv_skills or not job_skills:
        return None
    
    # ── Embeddings ──────────────────────────────────────────
    cv_embs  = job_matcher.model.encode([s.lower() for s in cv_skills],  show_progress_bar=False)
    job_embs = job_matcher.model.encode([s.lower() for s in job_skills], show_progress_bar=False)
    
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim
    import numpy as np
    
    # ── Similarité embedding texte complet ──────────────────
    cv_text_emb  = job_matcher.model.encode([cv_text],  show_progress_bar=False)
    job_text_emb = job_matcher.model.encode([job_text], show_progress_bar=False)
    global_embedding_sim = float(cos_sim(cv_text_emb, job_text_emb)[0][0])
    
    # ── Matching skill par skill ─────────────────────────────
    similarities = []
    covered_strict   = 0  # >= threshold
    covered_moderate = 0  # >= 40%
    exact_matches    = 0
    
    for i, job_skill in enumerate(job_skills):
        best_sim = 0
        
        # Exact match
        for cv_skill in cv_skills:
            if cv_skill.lower() == job_skill.lower():
                best_sim = 100.0
                exact_matches += 1
                break
        
        # Semantic match
        if best_sim < 100:
            sims = cos_sim([job_embs[i]], cv_embs)[0] * 100
            best_sim = float(np.max(sims))
        
        similarities.append(best_sim)
        if best_sim >= threshold: covered_strict += 1
        if best_sim >= 40:        covered_moderate += 1
    
    sim_array = np.array(similarities)
    
    # ── Features techniques / soft séparées ─────────────────
    # Coverage technique uniquement
    tech_covered = 0
    if job_technical:
        job_tech_embs = job_matcher.model.encode([s.lower() for s in job_technical], show_progress_bar=False)
        cv_tech_embs  = job_matcher.model.encode([s.lower() for s in cv_technical],  show_progress_bar=False) if cv_technical else None
        
        for i, job_skill in enumerate(job_technical):
            best_sim = 0
            for cv_skill in cv_technical:
                if cv_skill.lower() == job_skill.lower():
                    best_sim = 100.0
                    break
            if best_sim < 100 and cv_tech_embs is not None:
                sims = cos_sim([job_tech_embs[i]], cv_tech_embs)[0] * 100
                best_sim = float(np.max(sims))
            if best_sim >= threshold:
                tech_covered += 1
    
    tech_coverage = (tech_covered / len(job_technical) * 100) if job_technical else 0
    
    return {
        # ✅ Coverage avec seuil strict (65%)
        'coverage_strict':    (covered_strict / len(job_skills) * 100),
        
        # ✅ Coverage avec seuil modéré (40%)
        'coverage_moderate':  (covered_moderate / len(job_skills) * 100),
        
        # ✅ Coverage technique uniquement (plus discriminant)
        'coverage_technical': tech_coverage,
        
        # ✅ Qualité moyenne des matchs
        'quality_mean':       float(sim_array.mean()),
        
        # ✅ Qualité max des matchs
        'quality_max':        float(sim_array.max()),
        
        # ✅ Qualité médiane (robuste aux outliers)
        'quality_median':     float(np.median(sim_array)),
        
        # ✅ % de skills avec sim > 80% (vrais experts)
        'high_match_ratio':   float((sim_array >= 80).mean() * 100),
        
        # ✅ % de skills avec sim < 30% (vrais manquants)
        'missing_ratio':      float((sim_array < 30).mean() * 100),
        
        # ✅ Exact matches (très discriminant!)
        'exact_match_count':  exact_matches,
        'exact_match_ratio':  (exact_matches / len(job_skills) * 100),
        
        # ✅ Embedding similarité globale texte entier
        'global_embedding_sim': global_embedding_sim,
        
        # ✅ Nombre de skills
        'nb_cv_skills':       len(cv_skills),
        'nb_job_skills':      len(job_skills),
        'nb_cv_technical':    len(cv_technical),
        'nb_job_technical':   len(job_technical),
        
        # ✅ Ratio CV skills / Job skills (expérience relative)
        'skill_ratio':        len(cv_skills) / max(len(job_skills), 1),
    }

# Test rapide sur 3 exemples
print("🧪 Test compute_rich_features\n")
for score, label in [(0.0, 'No Fit'), (0.5, 'Partial Fit'), (1.0, 'Perfect Fit')]:
    sample = df[df['score_target'] == score].iloc[0]
    feats = compute_rich_features(str(sample['resume']), str(sample['job_description']))
    print(f"{'='*50}")
    print(f"Classe : {label}")
    if feats:
        for k, v in feats.items():
            print(f"  {k:<25} : {v:.2f}")

🧪 Test compute_rich_features

Classe : No Fit
  coverage_strict           : 25.00
  coverage_moderate         : 87.50
  coverage_technical        : 33.33
  quality_mean              : 64.28
  quality_max               : 100.00
  quality_median            : 62.46
  high_match_ratio          : 25.00
  missing_ratio             : 12.50
  exact_match_count         : 2.00
  exact_match_ratio         : 25.00
  global_embedding_sim      : 0.54
  nb_cv_skills              : 14.00
  nb_job_skills             : 8.00
  nb_cv_technical           : 3.00
  nb_job_technical          : 3.00
  skill_ratio               : 1.75
Classe : Partial Fit
  coverage_strict           : 0.00
  coverage_moderate         : 100.00
  coverage_technical        : 0.00
  quality_mean              : 47.71
  quality_max               : 47.71
  quality_median            : 47.71
  high_match_ratio          : 0.00
  missing_ratio             : 0.00
  exact_match_count         : 0.00
  exact_match_ratio         : 0.00
  globa

In [18]:
# ============================================================
# CELLULE 10 : Calculer features sur 30 samples par classe
# ============================================================

print("🔄 Calcul features enrichies sur 30 samples par classe...\n")

all_features = []

for score, label in [(0.0, 'No Fit'), (0.5, 'Partial Fit'), (1.0, 'Perfect Fit')]:
    samples = df[df['score_target'] == score].head(30)
    
    for _, row in samples.iterrows():
        feats = compute_rich_features(str(row['resume']), str(row['job_description']))
        if feats:
            feats['score_target'] = score
            all_features.append(feats)

df_rich = pd.DataFrame(all_features)

print("📊 FEATURES MOYENNES PAR CLASSE :\n")
feature_cols = [c for c in df_rich.columns if c != 'score_target']

print(f"{'Feature':<25} {'No Fit':>10} {'Partial':>10} {'Perfect':>10} {'Discriminant?':>15}")
print("-" * 75)

for feat in feature_cols:
    no_fit  = df_rich[df_rich['score_target'] == 0.0][feat].mean()
    partial = df_rich[df_rich['score_target'] == 0.5][feat].mean()
    perfect = df_rich[df_rich['score_target'] == 1.0][feat].mean()
    
    # Discriminant si Perfect >> No Fit
    gap = abs(perfect - no_fit)
    disc = "✅ OUI" if gap > 10 else ("⚠️ MOYEN" if gap > 5 else "❌ NON")
    
    print(f"{feat:<25} {no_fit:>10.1f} {partial:>10.1f} {perfect:>10.1f} {disc:>15}")

🔄 Calcul features enrichies sur 30 samples par classe...

📊 FEATURES MOYENNES PAR CLASSE :

Feature                       No Fit    Partial    Perfect   Discriminant?
---------------------------------------------------------------------------
coverage_strict                 20.2       23.0       33.5           ✅ OUI
coverage_moderate               52.7       54.6       66.6           ✅ OUI
coverage_technical              16.4       21.8       32.9           ✅ OUI
quality_mean                    50.4       51.6       60.5           ✅ OUI
quality_max                     87.2       82.6       91.8           ❌ NON
quality_median                  46.3       46.9       56.2        ⚠️ MOYEN
high_match_ratio                17.7       20.8       31.2           ✅ OUI
missing_ratio                   18.7       21.7       12.9        ⚠️ MOYEN
exact_match_count                1.6        1.4        3.3           ❌ NON
exact_match_ratio               17.7       20.8       31.0           ✅ OUI
global_